# 04 GMM

[![Code License: MIT](https://img.shields.io/badge/Code%20License-MIT-yellow.svg)](../LICENSE)
[![Content License: CC BY 4.0](https://img.shields.io/badge/Content%20License-CC%20BY%204.0-blue.svg)](https://creativecommons.org/licenses/by/4.0/)

In [ ]:
# === Environment Setup ===
import os, sys, math, time, random, json, textwrap, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import chi2
import statsmodels.api as sm
from statsmodels.sandbox.regression.gmm import IV2SLS
from IPython.display import display, Markdown

# --- Configuration ---
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 12, 'figure.figsize': (11, 7), 'figure.dpi': 130})
np.set_printoptions(suppress=True, linewidth=120, precision=4)

print("Environment initialized for Generalized Method of Moments.")

# The Lens: The Method of Moments

**What economic problem are we solving?**
Most economic models—whether supply and demand, rational expectations, or asset pricing—do not imply a full probability distribution for the data (like MLE requires). Instead, they imply specific conditions that must hold on average: supply equals demand, forecast errors are zero, or asset returns are unpredictable. We need a way to estimate parameters using *only* these theoretical conditions, without making strong, potentially false assumptions about the distribution of errors.

**Why do we need this method?**
The **Generalized Method of Moments (GMM)** is the unifying framework for this. It finds the parameters that make the sample moments (what we see in data) match the theoretical moments (what the model predicts) as closely as possible. OLS, IV, and MLE are all special cases of GMM. Understanding GMM unlocks the ability to estimate complex, non-linear models where other methods fail.

### Table of Contents
1.  [Introduction: The Power of Moment Conditions](#1.-Introduction:-The-Power-of-Moment-Conditions)
2.  [From Moments to an Objective Function](#2.-From-Moments-to-an-Objective-Function)
    *   [The GMM Criterion Function](#The-GMM-Criterion-Function)
    *   [The Role of the Weighting Matrix](#The-Role-of-the-Weighting-Matrix)
3.  [The Two-Step Efficient GMM Estimator](#3.-The-Two-Step-Efficient-GMM-Estimator)
4.  [Asymptotic Properties of the GMM Estimator](#4.-Asymptotic-Properties-of-the-GMM-Estimator)
5.  [Implementation: A Reusable GMM Tool](#5.-Implementation:-A-Reusable-GMM-Tool)
6.  [Example 1: Instrumental Variables as GMM](#6.-Example-1:-Instrumental-Variables-as-GMM)
7.  [Example 2: Non-Linear GMM for Asset Pricing](#7.-Example-2:-Non-Linear-GMM-for-Asset-Pricing)
8.  [Hypothesis Testing: The J-Test](#8.-Hypothesis-Testing:-The-J-Test)
9.  [Summary](#9.-Summary)
10. [Exercises](#10.-Exercises)

### 1. Introduction: The Power of Moment Conditions

The **Generalized Method of Moments (GMM)**, formalized by Lars Peter Hansen in his seminal 1982 paper, is one of the most important and versatile estimation frameworks in modern econometrics. Its power lies in its generality. While Maximum Likelihood Estimation (MLE) requires specifying the entire probability distribution of the data, GMM requires only that we specify a set of **moment conditions** that should hold true at the population level.

Many estimation problems can be framed in this way. The core idea is simple but profound:

> Economic theory often provides orthogonality or moment conditions, which state that the expected value of some function of the data and parameters is zero. The GMM principle is to choose the parameter estimates that make the sample analogue of these population moment conditions as close to zero as possible.

This framework is incredibly powerful because it unifies many well-known estimators under a single conceptual umbrella:
- **Ordinary Least Squares (OLS)** is a GMM estimator where the moment conditions are that the regressors are orthogonal to the error term.
- **Instrumental Variables (IV)** is a GMM estimator where the moment conditions are that the instruments are orthogonal to the error term.
- **Maximum Likelihood (MLE)** can be shown to be a GMM estimator where the moment conditions are given by the score vector.

### 2. From Moments to an Objective Function

Let's formalize the GMM principle. Suppose our economic theory implies a set of $r$ population moment conditions:

$$ E[g(W_i, \theta_0)] = 0 $$

where $W_i$ is a vector of observed data for individual $i$, $\theta_0$ is the $k \times 1$ vector of true parameters we want to estimate, and $g(\cdot)$ is a vector-valued function with $r$ elements. For the model to be identified, we need at least as many moment conditions as parameters, so $r \ge k$.

The sample analogue of the population moment condition is the sample average:

$$ g_N(\theta) = \frac{1}{N} \sum_{i=1}^N g(W_i, \theta) $$

Due to sampling variation, $g_N(\theta_0)$ will not be exactly zero, even at the true parameter value. The GMM approach is to find the parameter vector $\hat{\theta}$ that makes $g_N(\hat{\theta})$ as "close" to the zero vector as possible.

#### The GMM Criterion Function

We measure this "closeness" using a quadratic form. The GMM estimator $\hat{\theta}_{GMM}$ is the value of $\theta$ that minimizes the following objective function. This function is a weighted sum of the squared sample moments.

$$ J(\theta, W) = N \cdot g_N(\theta)' W g_N(\theta) $$

where $W$ is an $r \times r$ symmetric, positive definite **weighting matrix**. This matrix determines how we penalize deviations from the different moment conditions.

#### The Role of the Weighting Matrix

Why do we need a weighting matrix? The different moment conditions in $g_N(\theta)$ may have different variances and may be correlated with each other. A good weighting matrix should give more weight to the moment conditions that are estimated more precisely (i.e., have smaller variance) and account for the correlation between them.

Hansen (1982) showed that the **asymptotically efficient** GMM estimator (the one with the smallest asymptotic variance) is obtained by using a weighting matrix that is a consistent estimate of the inverse of the variance-covariance matrix of the moment conditions, $S$.

$$ S = Var(\sqrt{N} g_N(\theta_0)) = E[g(W_i, \theta_0)g(W_i, \theta_0)'] $$

The optimal weighting matrix is therefore $W_{opt} = S^{-1}$. The problem is that $S$ itself depends on the unknown true parameters $\theta_0$. This leads to a sequential estimation procedure.

### 3. The Two-Step Efficient GMM Estimator

The solution to the problem of the optimal weighting matrix depending on the parameters is the now-standard **Two-Step GMM** procedure.

1.  **Consistent First Step:** Estimate $\theta$ using a sub-optimal but valid weighting matrix (like the identity matrix, $W=I$). Call this $\hat{\theta}_1$.
2.  **Estimate S:** Use $\hat{\theta}_1$ to estimate the covariance matrix of moments, $\hat{S}$.
3.  **Efficient Second Step:** Estimate $\theta$ again, minimizing the GMM objective using the optimal weighting matrix $\hat{W} = \hat{S}^{-1}$. This yields the efficient estimator $\hat{\theta}_{GMM}$.

### 4. Asymptotic Properties of the GMM Estimator

Under a set of regularity conditions, the efficient Two-Step GMM estimator has the following properties:

1.  **Consistency:** $\hat{\theta}_{GMM} \xrightarrow{p} \theta_0$
2.  **Asymptotic Normality:**
    $$ \sqrt{N}(\hat{\theta}_{GMM} - \theta_0) \xrightarrow{d} N(0, V_{GMM}) $$

The asymptotic variance-covariance matrix is given by $V_{GMM} = (G'S^{-1}G)^{-1}$, where $G$ is the matrix of expected derivatives of the moments.

In [ ]:
### Implementation: A Reusable GMM Tool

class GMMEstimator:
    """
    A simple class to perform Two-Step GMM estimation.
    
    Args:
        moment_conditions (callable): A function that takes (theta, data) and returns an (N x r) matrix of moment contributions.
        data (dict): A dictionary containing the data arrays needed by the moment conditions function.
        param_names (list): A list of names for the parameters (theta).
    """
    def __init__(self, moment_conditions, data, param_names=None):
        self.moment_conditions = moment_conditions
        self.data = data
        self.n_obs = next(iter(data.values())).shape[0]
        self.param_names = param_names
        self.gmm_params = None
        self.gmm_vcov = None
        self.j_stat = None
        self.j_pval = None

    def fit(self, start_params):
        """Performs the two-step GMM estimation."""
        n_params = len(start_params)
        if self.param_names is None:
            self.param_names = [f'p{i}' for i in range(n_params)]
        
        # --- Step 1: Initial consistent estimation with Identity Weighting Matrix ---
        W1 = np.identity(self.moment_conditions(start_params, self.data).shape[1])
        
        def criterion_fn(theta, W):
            g = self.moment_conditions(theta, self.data)
            g_mean = np.mean(g, axis=0)
            return self.n_obs * (g_mean.T @ W @ g_mean)
            
        res1 = minimize(criterion_fn, start_params, args=(W1,), method='BFGS')
        theta1 = res1.x

        # --- Step 2: Efficient GMM with Optimal Weighting Matrix ---
        g1 = self.moment_conditions(theta1, self.data)
        S_hat = (g1.T @ g1) / self.n_obs
        W2 = np.linalg.inv(S_hat)

        res2 = minimize(criterion_fn, theta1, args=(W2,), method='BFGS')
        self.gmm_params = res2.x
        self.j_stat = res2.fun
        
        # --- Calculate Standard Errors ---
        # Numerical differentiation for G
        epsilon = 1e-6
        G_hat = np.zeros((g1.shape[1], n_params))
        for i in range(n_params):
            theta_plus = self.gmm_params.copy()
            theta_plus[i] += epsilon
            g_plus = np.mean(self.moment_conditions(theta_plus, self.data), axis=0)
            
            theta_minus = self.gmm_params.copy()
            theta_minus[i] -= epsilon
            g_minus = np.mean(self.moment_conditions(theta_minus, self.data), axis=0)
            
            G_hat[:, i] = (g_plus - g_minus) / (2 * epsilon)

        # Variance-covariance matrix
        V_hat = np.linalg.inv(G_hat.T @ W2 @ G_hat)
        self.gmm_vcov = V_hat / self.n_obs
        
        # J-test p-value
        dof = g1.shape[1] - n_params
        if dof > 0:
            self.j_pval = chi2.sf(self.j_stat, df=dof)
        
        return self

    def summary(self):
        """Prints a summary of the GMM results."""
        if self.gmm_params is None:
            print("Model has not been fitted yet.")
            return
        
        se = np.sqrt(np.diag(self.gmm_vcov))
        z_stats = self.gmm_params / se
        p_values = chi2.sf(z_stats**2, df=1)
        
        results_df = pd.DataFrame({
            'Estimate': self.gmm_params,
            'Std. Error': se,
            'Z-statistic': z_stats,
            'P-value': p_values
        }, index=self.param_names)
        
        print("Two-Step GMM Results")
        print(f"N. of Observations: {self.n_obs}")
        display(results_df.round(4))
        
        if self.j_stat is not None and self.j_pval is not None:
            print("\nOveridentification Test (Hansen's J):")
            print(f"J-statistic: {self.j_stat:.4f}")
            print(f"P-value: {self.j_pval:.4f}")

### 6. Example 1: Instrumental Variables as GMM

Our first example shows that standard IV is a special case of GMM. 

Consider the model:
$$ y_i = \mathbf{x}_i'\beta + u_i $$

where some regressors are endogenous. We have instruments $\mathbf{z}_i$ such that $E[\mathbf{z}_i u_i] = 0$. This gives us our moment conditions:

$$ E[\mathbf{z}_i (y_i - \mathbf{x}_i'\beta)] = 0 $$

In [ ]:
### IV as GMM: Simulation

# 1. Simulate data with an endogenous regressor
rng = np.random.default_rng(seed=1234)
N = 1000
true_beta = np.array([0.5, 2.0])

# Instruments (exogenous)
z1 = rng.standard_normal(N)
z2 = rng.standard_normal(N)
Z = np.vstack([z1, z2]).T

# Disturbance term 'v' for the endogenous regressor
v = 0.7 * z1 + 0.3 * z2 + rng.standard_normal(N)

# Error term 'u' for the main equation
u = 0.5 * v + rng.standard_normal(N) # u and v are correlated

# Endogenous regressor 'x'
x = 1 + 0.5 * v

# Dependent variable 'y'
X = sm.add_constant(x)
y = X @ true_beta + u

# 2. Estimate with our GMM class
# The instruments for the model include the constant and z1, z2
instruments = sm.add_constant(Z)
data_iv = {'y': y, 'X': X, 'Z': instruments}

def iv_moment_conditions(beta, data):
    """Returns the (N x r) matrix of moment contributions for IV."""
    y, X, Z = data['y'], data['X'], data['Z']
    u = y - X @ beta
    g = Z * u[:, np.newaxis] # Element-wise multiplication, broadcasting u
    return g

gmm_iv = GMMEstimator(iv_moment_conditions, data_iv, param_names=['const', 'x1'])
gmm_iv.fit(start_params=[0, 0])
print("Estimating the model using our GMMEstimator class:")
gmm_iv.summary()

# 3. Compare with a standard IV/2SLS estimator
print("\nComparing the results to Statsmodels' standard IV2SLS estimator:")
iv_sm = IV2SLS(y, X, instruments).fit()
print(iv_sm.summary())

### 7. Example 2: Non-Linear GMM for Asset Pricing

The true power of GMM shines in non-linear models. A classic application is the **consumption-based capital asset pricing model (CCAPM)**.

The core pricing equation (Euler equation) is:

$$ E_t[\beta \left(\frac{C_{t+1}}{C_t}\right)^{-\gamma} R_{i, t+1} - 1] = 0 $$

This implies that any variable $Z_t$ in the time-$t$ information set is orthogonal to the pricing error. This provides the moment conditions to estimate $(\beta, \gamma)$.

In [ ]:
### Non-Linear GMM: CCAPM Estimation

# 1. Load or simulate asset pricing data
rng = np.random.default_rng(seed=42)
T = 500

# True parameters
true_params = {'beta': 0.99, 'gamma': 2.5}

# Simulate data
cons_growth = np.exp(rng.normal(0.02, 0.02, T))
asset_return = np.exp(rng.normal(0.06, 0.15, T))

# Create lagged instruments
Z_t = np.vstack([
    np.ones(T-1),
    cons_growth[:-1],
    asset_return[:-1]
]).T

# Align data
C_t1_over_Ct = cons_growth[1:]
R_t1 = asset_return[1:]

data_ccapm = {'c_growth': C_t1_over_Ct, 'returns': R_t1, 'instruments': Z_t}

# 2. Define the non-linear moment conditions
def ccapm_moment_conditions(theta, data):
    beta, gamma = theta[0], theta[1]
    c_growth = data['c_growth']
    returns = data['returns']
    Z = data['instruments']
    
    # Pricing error
    pricing_error = (beta * c_growth**(-gamma) * returns) - 1
    
    # Moment contributions g_i = Z_i * error_i
    g = Z * pricing_error[:, np.newaxis]
    return g

# 3. Estimate with our GMM class
gmm_ccapm = GMMEstimator(ccapm_moment_conditions, data_ccapm, param_names=['beta', 'gamma'])
gmm_ccapm.fit(start_params=[0.95, 2.0])
print("Estimating the CCAPM parameters using our GMMEstimator:")
gmm_ccapm.summary()

### 8. Hypothesis Testing: The J-Test

When we have more moment conditions than parameters ($r > k$), the model is **overidentified**. We can test the validity of the moment conditions using Hansen's **J-test**.

The J-statistic is the minimized value of the GMM objective function. Under the null hypothesis that the model is correctly specified, it follows a chi-squared distribution: $ J \xrightarrow{d} \chi^2_{r-k} $.

- **Small J-statistic:** Do not reject null. The sample moments are close to zero, supporting the model.
- **Large J-statistic:** Reject null. The model is misspecified.

# Summary

**What did we learn?**
- **Moment Conditions:** GMM estimates parameters by matching sample moments to population moments derived from theory.
- **Generality:** OLS, IV, and MLE are all special cases of GMM.
- **Weighting Matrix:** The efficient GMM estimator uses the inverse of the moment covariance matrix as weights.
- **J-Test:** The J-statistic tests the validity of overidentifying restrictions, providing a crucial check for model specification.

### 10. Exercises

1.  **Just-Identification:** What happens to the J-statistic when the model is exactly identified ($r=k$)? Explain both intuitively and by looking at the formula for the GMM estimator.

2.  **OLS as GMM:** Consider the classical linear model $y_i = \mathbf{x}_i'\beta + u_i$, where $E[\mathbf{x}_i u_i] = 0$. 
    a. Write down the moment conditions for this model.
    b. Write down the sample moment vector $g_N(\beta)$.
    c. For the just-identified case, the GMM estimator sets $g_N(\hat{\beta})=0$. Solve this equation for $\hat{\beta}$. Do you recognize the result?

3.  **Alternative Instruments:** In the IV-as-GMM example, we used `[const, z1, z2]` as our instruments. What would happen if you only used `[const, z1]`? The model would be just-identified. Re-run the estimation with this smaller set of instruments. How do the parameter estimates and standard errors change? What is the J-statistic now?

4.  **Continuously Updated GMM (CUE):** Instead of the two-step approach, one could minimize a criterion that updates the weighting matrix at every iteration: $J_{CUE}(\theta) = g_N(\theta)' [S(\theta)]^{-1} g_N(\theta)$. This is the Continuously Updated GMM estimator. Discuss the potential advantages and disadvantages of this approach compared to the two-step method. (Hint: Think about statistical efficiency vs. computational cost).